# torch.nn.functional：激活函数与损失函数

PyTorch 神经网络有两套常用接口：**`nn.Module`（层/模块）** 与 **`F = torch.nn.functional`（函数）**。下文先讲二者关系，再进入激活函数与损失函数。

**学完应能回答：**
1. `F.relu(x)` 和 `nn.ReLU()(x)` 有什么区别？什么时候用哪个？
2. 为什么几乎所有深度网络都要非线性激活？
3. 回归用 MSE 还是 Huber？分类用 CE 还是 BCE？logit 版和概率版差在哪？

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

GELU_BETA = 1.702


def swish(x, beta=1.0):
    """Swish: x * σ(βx)。β=1 时即 SiLU。"""
    return x * torch.sigmoid(beta * x)


def plot_2x2(panels, title):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, p in zip(axes.flat, panels):
        for name, y in p["curves"].items():
            ax.plot(p["x"], y, lw=2, ls="--" if name in p.get("dashed", ()) else "-", label=name)
        ax.axhline(0, c="gray", ls="--", lw=0.8)
        ax.axvline(0, c="gray", ls="--", lw=0.8)
        ax.set(title=p["title"], xlabel=p["xlabel"], ylabel=p.get("ylabel", "f(x)"))
        if "ylim" in p:
            ax.set_ylim(*p["ylim"])
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    return fig


%matplotlib inline

## 0. `F` 与 `nn.Module` 是什么关系？

PyTorch 的 `torch.nn` 里有两层抽象：

| | **`nn.Module`** | **`F = torch.nn.functional`** |
|---|-----------------|----------------------------------|
| 形态 | 类，实例化后当「层」用 | 纯函数，直接调 |
| 参数 | 可持有 **可学习参数**（`weight`、`bias`） | **无参数** |
| 状态 | 有 `train()` / `eval()`，影响 Dropout、BN 等 | 无内部状态 |
| 典型用途 | 搭 `nn.Sequential`、写 `forward` | `forward` 内部调用，或临时算一下 |

**核心关系：`nn.Module` 是对 `F` 的封装。** 很多层的 `forward` 就是在调对应的 `F` 函数：

```
nn.ReLU().forward(x)      →  F.relu(x)
nn.Linear(...).forward(x) →  F.linear(x, weight, bias)
nn.CrossEntropyLoss(...)(pred, tgt) → F.cross_entropy(pred, tgt, ...)
```

### 什么时候用哪个？

**用 `nn.Module`（层）** — 需要参数，或要进模型结构：
```python
self.fc = nn.Linear(768, 10)   # 有 weight/bias，会自动注册、保存、优化
self.act = nn.GELU()
```

**用 `F`（函数）** — 无参数运算，或在自定义 `forward` 里灵活组合：
```python
def forward(self, x):
    x = self.fc(x)              # 有参数的用 Module
    x = F.gelu(x)               # 激活直接用 F 也行
    x = F.dropout(x, p=0.1, training=self.training)  # Dropout 需手动传 training
```

### 几个容易混的点

1. **等价不等于完全一样**：`F.relu(x)` 与 `nn.ReLU()(x)` 数值相同；但 `nn.Dropout` 会记住训练/推理模式，`F.dropout` 必须自己传 `training=`。
2. **有参数的层没法用纯 `F` 替代整个层**：`F.linear` 仍要传入 `weight`、`bias`，参数管理得自己做；所以线性层几乎总是 `nn.Linear`。
3. **损失函数两边都有**：`nn.CrossEntropyLoss()` 是带 `reduction` 等配置的 Module；`F.cross_entropy(..., reduction='mean')` 是一次性函数调用。训练脚本里两种写法都常见。
4. **本 notebook 为何多用 `F`**：激活函数和损失函数本身**没有可学习参数**，用 `F` 写公式最直观；搭完整模型时仍会用 `nn.Module` 包一层。

In [ ]:
import torch.nn as nn

x = torch.tensor([-1.0, 0.0, 2.0])

# 无参数层：Module 与 F 数值一致
print("ReLU  Module:", nn.ReLU()(x).tolist())
print("ReLU  F:     ", F.relu(x).tolist())

# 有参数层：Module 帮你管 weight/bias；F 要手动传
linear = nn.Linear(1, 1)
x_col = x.unsqueeze(1)
out_module = linear(x_col)
out_func = F.linear(x_col, linear.weight, linear.bias)
print("Linear 相同?", torch.allclose(out_module, out_func))

# 损失：Module 封装 F，默认 reduction='mean'
pred = torch.randn(3, 5)
tgt = torch.tensor([1, 0, 4])
print("CE  Module:", nn.CrossEntropyLoss()(pred, tgt).item())
print("CE  F:     ", F.cross_entropy(pred, tgt).item())

## 1. 激活函数

### 1.1 为什么需要它？

- 多层线性变换叠在一起仍是线性：$W_2(W_1 x)= (W_2 W_1)x$。**没有非线性就表达不出 XOR、边缘、语义等复杂模式。**
- 激活函数把线性投影「掰弯」，网络才有分层特征能力。
- 选型时主要看：是否饱和（梯度消失）、负半轴如何处理（死神经元）、是否平滑（优化友好）、计算成本。

### 1.2 速查表（公式 + 场景）

| 函数 | 公式 | 典型场景 | 注意点 |
|------|------|----------|--------|
| Sigmoid | $\sigma(x)=\dfrac{1}{1+e^{-x}}$ | 二分类**输出层**概率；门控（LSTM） | 深层隐层少用：两端饱和 → 梯度消失；输出非零均值 |
| Tanh | $\tanh(x)=2\sigma(2x)-1$ | RNN/LSTM 隐状态；需要 $[-1,1]$ | 仍有饱和，但零均值，比 Sigmoid 稍好 |
| ReLU | $\max(0,x)$ | CNN / 早期 MLP 默认 | 简单快；负半轴恒 0 → **死 ReLU** |
| Leaky ReLU | $x>0$ 取 $x$，否则 $\alpha x$ | 想缓解死神经元时 | $\alpha$ 常用 0.01~0.2 |
| ELU | $x>0$ 取 $x$，否则 $\alpha(e^x-1)$ | 希望负半轴平滑、均值近 0 | 比 ReLU 稍慢 |
| GELU | $x\Phi(x)$ | **Transformer / BERT / GPT 标配** | 平滑、概率加权；可用 $x\sigma(1.702x)$ 近似 |
| SiLU / Swish | $x\sigma(\beta x)$，$\beta=1$ 即 SiLU | EfficientNet、部分现代 CNN/MLP | 非单调；自门控 |
| Softmax | $p_i=e^{z_i}/\sum_j e^{z_j}$ | 多分类**输出层**；注意力权重 | 不是逐元素激活，对一整组 logit 归一化 |

### 1.3 选型直觉

1. **隐层默认**：现代 NLP/多模态 → **GELU**；视觉 CNN 仍常见 **ReLU / SiLU**。
2. **输出层看任务**：多类互斥 → Softmax；二分类概率 → Sigmoid；回归 → 通常**不加**激活（或按值域加）。
3. **Sigmoid/Tanh 别堆在深层隐层**：饱和区导数接近 0，深层很难训。
4. **ReLU「死掉」**：某单元长期输出 0 且梯度为 0，可用 Leaky ReLU / ELU / GELU，或检查学习率、初始化。

In [ ]:
x = torch.linspace(-6, 6, 600)
logits = torch.zeros(x.numel(), 3)
logits[:, 0] = x

plot_2x2([
    {"title": "有界 / S 形（注意饱和）", "x": x, "xlabel": "x", "ylim": (-1.2, 1.2),
     "curves": {"Sigmoid": torch.sigmoid(x), "Tanh": torch.tanh(x),
                "2σ(2x)−1": 2 * torch.sigmoid(2 * x) - 1,
                "Softmax p₀": F.softmax(logits, -1)[:, 0]},
     "dashed": ("2σ(2x)−1",)},
    {"title": "ReLU 家族（负半轴策略）", "x": x, "xlabel": "x", "ylim": (-2, 6),
     "curves": {"ReLU": F.relu(x), "Leaky ReLU": F.leaky_relu(x, 0.1), "ELU": F.elu(x)}},
    {"title": "GELU & Swish（现代默认）", "x": x, "xlabel": "x", "ylim": (-1.5, 6),
     "curves": {"GELU": F.gelu(x), "Swish(β=1.702)": swish(x, GELU_BETA), "SiLU(β=1)": swish(x)},
     "dashed": ("Swish(β=1.702)",)},
    {"title": "Swish 的 β（越大越像 ReLU）", "x": x, "xlabel": "x", "ylim": (-1.5, 6),
     "curves": {f"β={b}": swish(x, b) for b in (0.5, 1.0, 1.702, 2.0)}},
], "Activation Functions")
plt.show()

### 1.4 读图要点

- **左上**：Sigmoid / Tanh 在 $|x|$ 大时几乎水平 → 梯度≈0（饱和）。虚线验证 $\tanh(x)=2\sigma(2x)-1$。
- **右上**：ReLU 负半轴砍死；Leaky/ELU 留一条「活路」。
- **左下**：GELU ≈ Swish($\beta=1.702$)；都比 ReLU 更平滑，负半轴略有回弹。
- **右下**：$\beta\uparrow$，Swish 越接近 ReLU。

## 2. 损失函数

### 2.1 先分清任务

| 任务 | 输出含义 | 常用损失 |
|------|----------|----------|
| 回归（连续值） | $\hat{y}\in\mathbb{R}$ | MSE / MAE / Huber |
| 二分类 | 概率 $p$ 或 logit $z$ | BCE / BCEWithLogits |
| 多分类（互斥） | 各类 logit | CrossEntropy（内部 Softmax） |
| 多标签（可不互斥） | 每类独立概率 | 每类 BCEWithLogits |

记误差 $e=y-\hat{y}$，二分类标签 $y\in\{0,1\}$，logit $z$（$p=\sigma(z)$）。

### 2.2 速查表（公式 + 场景）

| 函数 | 公式 | 典型场景 | 注意点 |
|------|------|----------|--------|
| MSE | $L=e^2$ | 回归默认；误差大致高斯 | 对离群点**很敏感** |
| MAE | $L=\|e\|$ | 有离群点、更看重中位数 | 零点不可导；大误差时梯度恒定 |
| Huber / SmoothL1 | $\|e\|\le\delta$ 用二次，否则线性 | 检测框回归、鲁棒回归 | 兼顾 MSE 与 MAE |
| BCE | $-[y\log p+(1-y)\log(1-p)]$ | 二分类 / 多标签（输入已是概率） | $p$ 贴近 0/1 时易数值不稳 |
| BCEWithLogits | 同上，但输入 logit | **推荐的二分类写法** | 内部合并 sigmoid+log，更稳 |
| CrossEntropy | $-\log\dfrac{e^{z_y}}{\sum_j e^{z_j}}$ | 多类互斥分类 | 输入是 **logit**，不要先 Softmax |
| Hinge | $\max(0, 1-y\cdot s)$，$y\in\{\pm1\}$ | SVM 风格间隔分类 | 更关心间隔，不是概率校准 |

### 2.3 BCE vs BCEWithLogits（必懂）

- **同一损失**，只是接口不同：
  - `binary_cross_entropy`：输入概率 $p\in(0,1)$
  - `binary_cross_entropy_with_logits`：输入 logit $z$，内部做 $\sigma$
- 稳定形式：$\max(z,0)-zy+\log(1+e^{-\|z\|})$，避免先算 $\sigma$ 再 $\log$ 造成 underflow。
- **实践建议**：网络最后一层不要 Sigmoid，直接出 logit，用 `BCEWithLogitsLoss` / `F.binary_cross_entropy_with_logits`。

### 2.4 CrossEntropy 易错点

- `F.cross_entropy(logits, target)` ≡ `LogSoftmax + NLLLoss`。
- **不要**再对 `logits` 做 Softmax 再送进 CE，相当于 Softmax 两次，语义和梯度都错。
- `target` 可以是类别下标 `LongTensor`，也可以是类别概率分布（软标签 / 知识蒸馏）。

In [ ]:
# 数值验证：同一 (z, y)，两种 BCE 接口应几乎相等
logit, target = torch.tensor(2.0), torch.tensor(1.0)
print("BCEWithLogits:", F.binary_cross_entropy_with_logits(logit, target).item())
print("BCE:          ", F.binary_cross_entropy(torch.sigmoid(logit), target).item())

In [ ]:
e = torch.linspace(-3, 3, 600)
p = torch.linspace(0.001, 0.999, 600)
z = torch.linspace(-6, 6, 600)
zero = torch.zeros_like
logits = torch.zeros(z.numel(), 3)
logits[:, 0] = z

plot_2x2([
    {"title": "回归：离群点时谁更稳？", "x": e, "xlabel": "e = y−ŷ", "ylabel": "loss", "ylim": (0, 5),
     "curves": {"MSE": e**2, "MAE": e.abs(),
                "Huber": F.smooth_l1_loss(e, zero(e), beta=1.0, reduction="none")}},
    {"title": "BCE vs 概率 p", "x": p, "xlabel": "p", "ylabel": "loss", "ylim": (0, 7),
     "curves": {"y=1": F.binary_cross_entropy(p, torch.ones_like(p), reduction="none"),
                "y=0": F.binary_cross_entropy(p, torch.zeros_like(p), reduction="none")}},
    {"title": "BCEWithLogits vs logit z", "x": z, "xlabel": "z", "ylabel": "loss", "ylim": (0, 7),
     "curves": {"y=1": F.binary_cross_entropy_with_logits(z, torch.ones_like(z), reduction="none"),
                "y=0": F.binary_cross_entropy_with_logits(z, torch.zeros_like(z), reduction="none")}},
    {"title": "Hinge & CrossEntropy", "x": z, "xlabel": "score / logit", "ylabel": "loss", "ylim": (0, 7),
     "curves": {"Hinge(y=+1)": F.relu(1 - z), "Hinge(y=−1)": F.relu(1 + z),
                "CE(class 0)": F.cross_entropy(logits, torch.zeros(z.numel(), dtype=torch.long), reduction="none")}},
], "Loss Functions")
plt.show()

### 2.5 读图要点

- **左上**：大 $|e|$ 时 MSE 飙升（惩罚离群点狠）；MAE 线性；Huber 中间像 MSE、两头像 MAE。
- **右上 / 左下**：预测错得越自信（$p$ 或 $z$ 站错边），BCE 越大——分类损失在「惩罚错误自信」。
- **右下**：Hinge 在间隔足够大时损失为 0；CE 始终按概率校准，不会「间隔够了就不管」。

### 2.6 一句话选型

```
回归 + 干净数据     → MSE
回归 + 有离群点     → Huber / MAE
二分类 / 多标签     → BCEWithLogits（输出 logit）
多类互斥            → CrossEntropy（输出各类 logit，勿先 Softmax）
隐层激活            → GELU（Transformer）/ ReLU·SiLU（CNN）
输出概率（二分类）  → Sigmoid；多类 → Softmax（通常由 CE 内部完成）
```